In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import pdb_tools
import MDAnalysis as mda
import os
import wts
from pdbparser.pdbparser import pdbparser
import warnings
from Bio import BiopythonWarning
with warnings.catch_warnings():
    warnings.simplefilter('ignore', BiopythonWarning)
    import Bio
    from Bio import PDB
    from Bio.PDB.PDBParser import PDBParser
warnings.filterwarnings("ignore", category=UserWarning)
# You can also run file as "$python -W ignore check.py" to not have to deal with the unnecessary Warnings, but this will ignore
# all warnings, not just the ones you might not be interested in.
#warnings.filterwarnings("ignore", category=BioPythonWarning.PDBConstructionWarning)

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import pdb_tools
import MDAnalysis as mda
import os
import wts
from pdbparser.pdbparser import pdbparser
import warnings
from Bio import BiopythonWarning
with warnings.catch_warnings():
    warnings.simplefilter('ignore', BiopythonWarning)
    import Bio
    from Bio import PDB
    from Bio.PDB.PDBParser import PDBParser
warnings.filterwarnings("ignore", category=UserWarning)
# You can also run file as "$python -W ignore check.py" to not have to deal with the unnecessary Warnings, but this will ignore
# all warnings, not just the ones you might not be interested in.
#warnings.filterwarnings("ignore", category=BioPythonWarning.PDBConstructionWarning)



class PDBPrep():
    import pandas as pd
    import MDAnalysis as mda
    import wts

    def unique_ligands(self):
        self.verbose, self.cleanup, self.universes, self.hemes, self.residue_names = False, True, {}, {}, []
        self.read_pyDISH()
        for self.pdb_id in self.data['# PDB'].iloc[:1]:
            try:
                self.read_pdb()
                len_residue_list_before = len(self.residue_names)
            except FileNotFoundError:
                self.cleaner()
                continue
            try:
                self.get_axial()
                atoms_residues_in_structure = []
                lines = pdb_tools.files.read_file(f"PDB/{self.pdb_id}_axials.pdb")
                for line in lines:
                    line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
                    atoms_residues_in_structure.append(line_dict["resname"])
                for atoms_residue in atoms_residues_in_structure:
                    if atoms_residue not in self.residue_names:
                        self.residue_names.append(atoms_residue)
                len_residue_list_after = len(self.residue_names)
                if len_residue_list_before < len_residue_list_after:
                    with open("axial_ligands.txt", "w") as file:
                        file.write(str(self.residue_names))
            except IndexError:
                self.cleaner()
                continue
            except FileNotFoundError:
                self.cleaner()
                continue
            if self.cleanup:
                self.cleaner()
        return

    def error_catches(self):
        """Returns a ValueError when the column does not contain 4-letter alphanumerals."""
        bad_rows = []
        self.df['PDB str length'] = self.df['# PDB'].str.len()
        for index, row in self.df.iterrows():
            if row['PDB str length'] != 4:
                bad_rows.append(row)
        if bad_rows:
            for row in bad_rows:
                print(row)
            raise ValueError(f"PDB-ID length is not 4 for row_index {index}")

    def isolate_heme(self, filename):
        universe = mda.Universe(filename)
        heme_name_list = ["HEM", "HEA", "HEB", "HEC", "HEO"]
        atom_selection_string = f"resname {heme_name_list[0]} "
        for heme_name in heme_name_list[1:]:
            atom_selection_string += f"or resname {heme_name} "
        heme = universe.select_atoms(atom_selection_string)
        mda.Writer(f"PDB/{self.pdb_id}_heme.pdb").write(heme) #is pdb a string containing the pdb-id or a number?
        if heme.n_atoms==0:
            print(f"Current heme resnames: {heme_name_list}")
            raise ValueError(f"No atoms have been added to the Universe, because no atoms have been found with the resname identifiers for heme. Please add the identifier you find in the PDB to the heme identifying names. The PDB is {pdb}")
        return heme

    def convert_ent_to_pdb(self, ent_file_path, pdb_file_path):
        parser = PDBParser()
        structure = parser.get_structure('structure', ent_file_path)
        io = Bio.PDB.PDBIO()
        io.set_structure(structure)
        io.save(pdb_file_path)
        os.remove(ent_file_path)
        return pdb_file_path

    def HETATM_to_ATM(self):
        lines = pdb_tools.files.read_file(pdb_file=f"PDB/{self.pdb_id}_heme.pdb")
        lines_ = ["CRYST1\n"]
        if self.verbose:
            print(lines)
        for line in lines:
            line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
            pdb_tools.line_operations.exchange_HETATM_ATOM(line_dict=line_dict)
            line_ = pdb_tools.line_operations.create_line(line_dict=line_dict)
            lines_.append(line_)
        lines_.append("")
        if self.verbose:
            print(lines_)
        pdb_tools.files.write_file(file=f"PDB/{self.pdb_id}_heme_ATOM.pdb", lines=lines_)
        try:
            with open(f"PDB/{self.pdb_id}_heme_ATOM.pdb", 'r') as file:
                content = file.read()
            modified_content = content.replace(' E', 'FE')
            with open(f"PDB/{self.pdb_id}_heme_ATOM.pdb", 'w') as file:
                file.write(modified_content)
            if self.verbose:
                print(f"Occurrences of ' E' replaced with 'FE' in 'PDB/{self.pdb_id}_heme_ATOM.pdb'.")
        except FileNotFoundError:
            print(f"File 'PDB/{self.pdb_id}_heme_ATOM.pdb' not found.")
        return

    def pdb_to_xyz(self, input_pdb_file, output_xyz_file):
        with open(input_pdb_file, 'r') as pdb_file, open(output_xyz_file, 'w') as xyz_file:
            lines = pdb_file.readlines()
            num_atoms = 0
            atom_lines = []
            for line in lines:
                if line.startswith('ATOM') or line.startswith('HETATM'):
                    num_atoms += 1
                    atom_lines.append(line)
            xyz_file.write(str(num_atoms) + '\n')
            xyz_file.write('Converted from PDB to XYZ\n')
            for line in atom_lines:
                tokens = line.split()
                atom_symbol = tokens[2]
                atom_symbol = atom_symbol[0]
                #print(atom_symbol)
                x, y, z = float(tokens[6]), float(tokens[7]), float(tokens[8])
                xyz_file.write(f'{atom_symbol} {x:.6f} {y:.6f} {z:.6f}\n')
        return

    def concatenate_xyz(self, file_in1, file_in2, output):
        # Read the content of both XYZ files
        with open(file_in1, 'r') as xyz_file1, open(file_in2, 'r') as xyz_file2:
            lines1 = xyz_file1.readlines()
            lines2 = xyz_file2.readlines()
        # Calculate the total number of atoms
        num_atoms1 = int(lines1[0])
        num_atoms2 = int(lines2[0])
        total_atoms = num_atoms1 + num_atoms2
        # Concatenate the XYZ content and update the total atom count in the first line
        concatenated_lines = [f"{total_atoms}\n"] + lines1[1:] + lines2[2:]
        # Write the concatenated content to the output XYZ file
        with open(output, 'w') as output_file:
            output_file.writelines(concatenated_lines)

    def format_xyz_file(self):
        self.input_file, self.output_file = f"PDB/{self.pdb_id}.xyz", f"PDB/{self.pdb_id}.xyz"
        with open(self.input_file, 'r') as f:
            lines = f.readlines()
        atoms, coordinates = [], []
        for line in lines[2:]:
            parts = line.split()
            atoms.append(parts[0])
            coordinates.append([float(coord) for coord in parts[1:]])
        max_widths = [max(len(str(coord)) for coord in column) for column in zip(*coordinates)]
        with open(self.output_file, 'w') as f:
            f.write(f'{len(atoms)}\n')
            f.write(f'Heme structure file of {self.input_file} \n')
            for atom, coords in zip(atoms, coordinates):
                coord_str = ' '.join(f'{coord:{width}.6f}' for coord, width in zip(coords, max_widths))
                f.write(f'{atom:<2s} {coord_str}\n')
        return

    def xyz_to_pdb(self):
        assert os.path.isfile(f"PDB/{self.pdb_id}.xyz"), f"Unable to find xyz file 'PDB/{self.pdb_id}.xyz' of sorted heme atoms."
        with open(f"PDB/{self.pdb_id}.xyz", 'r') as file:
            lines = [l.strip() for l in file.readlines()]
        records = []
        for line in lines[2:]:
            element, x, y, z = line.split()
            records.append( { "record_name"       : 'ATOM',
                            "serial_number"     : len(records)+1,
                            "atom_name"         : element,
                            "location_indicator": '',
                            "residue_name"      : 'HEM',
                            "chain_identifier"  : 'H',
                            "sequence_number"   : len(records)+1,
                            "code_of_insertion" : '',
                            "coordinates_x"     : float(x),
                            "coordinates_y"     : float(y),
                            "coordinates_z"     : float(z),
                            "occupancy"         : 1.0,
                            "temperature_factor": 0.0,
                            "segment_identifier": '',
                            "element_symbol"    : element,
                            "charge"            : '',
                            } )
        pdb = pdbparser(filePath = None)
        pdb.records = records
        pdb.export_pdb(f"PDB/{self.pdb_id}_heme.pdb")
        return pdb

    def read_pyDISH(self):
        self.data = pd.read_csv('pyDISH_data.csv')
        self.data = self.data.drop('Unnamed: 0', axis=1)
        return

    def read_pdb(self):
        filename = PDB.PDBList().retrieve_pdb_file(self.pdb_id, pdir='PDB', file_format='pdb')
        filename = self.convert_ent_to_pdb(ent_file_path=f"PDB/pdb{self.pdb_id}.ent", pdb_file_path=f"PDB/{self.pdb_id}.pdb")
        #for offline testing:
        #filename=f"PDB/{self.pdb_id}.pdb"
        self.universes[self.pdb_id], self.hemes[self.pdb_id] = mda.Universe(filename), self.isolate_heme(filename)
        self.HETATM_to_ATM()
        return

    def get_PRAD(self):
        self.cleanup == False
        atom_res_dict = {
            " CAA" : "PRA ",
            " CBA" : "PRA ",
            " CGA" : "PRA ",
            " O1A" : "PRA ",
            " O2A" : "PRA ",
            " CAD" : "PRD ",
            " CBD" : "PRD ",
            " CGD" : "PRD ",
            " O1D" : "PRD ",
            " O2D" : "PRD ",
        }
        lines = pdb_tools.files.read_file(pdb_file=f"PDB/{self.pdb_id}_heme.pdb")
        lines_ = []
        for line in lines:
            line_dict = pdb_tools.line_operations.read_pdb_line(line)
            #catch PRA atoms
            #if line_dict["atom_name"] == " CAA":
            #    line_dict["resname"] = "PRA "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " CBA":
            #    line_dict["resname"] = "PRA "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " CGA":
            #    line_dict["resname"] = "PRA "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " O1A":
            #    line_dict["resname"] = "PRA "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " O2A":
            #    line_dict["resname"] = "PRA "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            ##Catch PRD atoms
            #if line_dict["atom_name"] == " CAD":
            #    line_dict["resname"] = "PRD "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " CBD":
            #    line_dict["resname"] = "PRD "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " CGD":
            #    line_dict["resname"] = "PRD "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " O1D":
            #    line_dict["resname"] = "PRD "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            #if line_dict["atom_name"] == " O2D":
            #    line_dict["resname"] = "PRD "
            #    line_ = pdb_tools.line_operations.create_line(line_dict)
            #    lines_.append(line_)
            try:
                if line_dict["resname"] in atom_res_dict:
                    line_dict["atom_name"] = atom_res_dict["resname"]
            except:
                print("Resname in PRA or PRD was not found. Writing error file. ")
                continue
        pdb_tools.files.write_file(file=f"PDB/{self.pdb_id}_prad.pdb", lines=lines_)
        return

    def get_axial(self):
        self.verbose, self.cleanup = False, True
        universe = self.universes[self.pdb_id]
        axials_find = universe.select_atoms("same resid as around 4 (type FE and resname HEM)")
        axials = axials_find.select_atoms("not resname HEM")
        try:
            axials.write(f"PDB/{self.pdb_id}_axials.pdb")
        except IndexError:
            return
        return

    def __try_clean(file):
        try:
            os.remove(file)
        except FileNotFoundError:
            pass
        return

    def cleaner(self):
        #PDBPrep.__try_clean(file=f"PDB/{self.pdb_id}.pdb")
        #PDBPrep.__try_clean(file=f"PDB/{self.pdb_id}_heme.pdb")
        PDBPrep.__try_clean(file=f"PDB/{self.pdb_id}_heme_ATOM.pdb")
        PDBPrep.__try_clean(file=f"PDB/{self.pdb_id}_prad.pdb")
        PDBPrep.__try_clean(file=f"PDB/{self.pdb_id}_axials.pdb")
        PDBPrep.__try_clean(file=f"PDB/{self.pdb_id}_heme_sorted.xyz")
        PDBPrep.__try_clean(file=f"PDB/{self.pdb_id}.xyz")
        return

    def fix_heme_atomtypes(self):
        lines, lines_ = pdb_tools.files.read_file(pdb_file=f"PDB/{self.pdb_id}_heme.pdb"), []
        atomtype_dict = {
            1  : " FE ",
            2  : " CHD",
            3  : " C4C",
            4  : " C3C",
            5  : " NC ",
            6  : " C2C",
            7  : " C1C",
            8  : " CHC",
            9  : " C4B",
            10 : " C3B",
            11 : " NB ",
            12 : " C2B",
            13 : " C1B",
            14 : " CHB",
            15 : " C4A",
            16 : " C3A",
            17 : " NA ",
            18 : " C2A",
            19 : " C1A",
            20 : " CHA",
            21 : " C4D",
            22 : " C3D",
            23 : " ND ",
            24 : " C2D",
            25 : " C1D"
        }
        for line in lines:
            line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
            line_dict["atom_name"] = atomtype_dict[int(line_dict["serial_no"].strip())]
            line_ = pdb_tools.line_operations.create_line(line_dict=line_dict)
            lines_.append(line_)
        pdb_tools.files.write_file(file=f"PDB/{self.pdb_id}_heme.pdb", lines=lines_)
        return

    def fix_heme(self):
        lines_ = []
        self.xyz_to_pdb()
        lines = pdb_tools.files.read_file(pdb_file=f"PDB/{self.pdb_id}_heme.pdb")
        for line in lines:
            line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
            line_dict["resi_no"] = "   1"
            #the following if condition is problematically hacky, and needs to be changed depending on the updates of pdb_tools.py by Nereu! -> open up Issue in pdb_tools.py repo. Also, pdbparser package???
            if line_dict["atom_name"].strip() == "FE":
                line_dict["elem_symb"] = "FE"
                line_dict["segment"] = "   "
            line_ = pdb_tools.line_operations.create_line(line_dict=line_dict)
            lines_.append(line_)
        pdb_tools.files.write_file(lines=lines_, file=f"PDB/{self.pdb_id}_heme.pdb")
        self.fix_heme_atomtypes()
        return

    def marry_pdbs(self):
        self.fix_heme()
        pdb_tools.operations.fuse_segments(pdb_files=[f"PDB/{self.pdb_id}_heme.pdb", f"PDB/{self.pdb_id}_prad.pdb", f"PDB/{self.pdb_id}_axials.pdb"], pdb_output=f"PDB/{self.pdb_id}_system.pdb")
        lines = pdb_tools.files.read_file(pdb_file=f"PDB/{self.pdb_id}_system.pdb")
        lines_, residues, seen = [], [], set()
        for line in lines:
            line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
            residues.append(line_dict["resname"])
        seen_add = seen.add
        residues = [k for k in residues if not (k in seen or seen_add(k))]
        axial1, axial2 = residues[3], residues[4]
        for line in lines:
            line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
            line_dict["chainID"], line_dict["segment"], line_dict["atom"] = "H", "HEME", "ATOM  "
            if line_dict["resname"] == "HEM":
                line_dict["resi_no"] = "   1"
            if line_dict["resname"].strip() == "PRA":
                line_dict["resi_no"] = "   2"
            if line_dict["resname"].strip() == "PRD":
                line_dict["resi_no"] = "   3"
            if line_dict["resname"] == axial1:
                line_dict["resi_no"] = "   4"
            if line_dict["resname"] == axial2:
                line_dict["resi_no"] = "   5"
            #check what atom_type and atom number are, then assign correct atom type
            # -> look up atom types in porphyrin rings in the CHARMM parameter sets
            # -> change topology files so that they have the
            #hacky fix for FE->E problem.
            if line_dict["atom_name"].strip() == "FE":
                line_dict["elem_symb"] = "FE"
            line_ = pdb_tools.line_operations.create_line(line_dict=line_dict)
            lines_.append(line_)
        pdb_tools.files.write_file(file=f"PDB/{self.pdb_id}_system.pdb", lines=lines_)
        #continue from here
        return

    def charmm_protonate(self):
        return

    def __init__(self):
        self.verbose, self.cleanup, self.universes, self.hemes, self.residue_names = False, True, {}, {}, []
        self.read_pyDISH()
        #self.unique_ligands()
        for self.pdb_id in self.data['# PDB'].iloc[:1]:
            self.read_pdb()
            # break if more than one heme
            # for now, since we need to do some further sorting before we can also catch multimers etc.
            if len(self.hemes[self.pdb_id].residues) != 1:
                print(len(self.hemes[self.pdb_id].residues))
                #os.remove(f"PDB/{self.pdb_id}.pdb")
                #os.remove(f"PDB/{self.pdb_id}_heme.pdb")
                self.cleaner()
                continue
            molecule = wts.porphyr(f"PDB/{self.pdb_id}_heme_ATOM.pdb", pdb_id=self.pdb_id)
            self.get_PRAD()
            self.get_axial()
            self.fix_heme()
            self.marry_pdbs()
            if self.cleanup:
                self.cleaner()
        return


if __name__ == "__main__":
    inst = PDBPrep()

In [5]:
def __try_clean(file):
    try:
        os.remove(file)
    except FileNotFoundError:
        pass
    return

def cleaner(pdb_id):
    __try_clean(file=f"PDB/{pdb_id}.pdb")
    __try_clean(file=f"PDB/{pdb_id}_heme.pdb")
    __try_clean(file=f"PDB/{pdb_id}_heme_ATOM.pdb")
    __try_clean(file=f"PDB/{pdb_id}_prad.pdb")
    __try_clean(file=f"PDB/{pdb_id}_axials.pdb")
    return

def get_axial(pdb_id, universes):
    verbose, cleanup = False, True
    universe = universes[pdb_id]
    axials_find = universe.select_atoms("same resid as around 4 (type FE and resname HEM)")
    axials = axials_find.select_atoms("not resname HEM")
    try:
        axials.write(f"PDB/{pdb_id}_axials.pdb")
    except IndexError:
        return
    return

def isolate_heme(pdb_id):
    universe = mda.Universe(f"PDB/{pdb_id}.pdb")
    heme_name_list = ["HEM", "HEA", "HEB", "HEC", "HEO"]
    atom_selection_string = f"resname {heme_name_list[0]} "
    for heme_name in heme_name_list[1:]:
        atom_selection_string += f"or resname {heme_name} "
    heme = universe.select_atoms(atom_selection_string)
    mda.Writer(f"PDB/{pdb_id}_heme.pdb").write(heme)
    if heme.n_atoms==0:
        print(f"Current heme resnames: {heme_name_list}")
        raise ValueError(f"No atoms have been added to the Universe, because no atoms have been found with the resname identifiers for heme. Please add the identifier you find in the PDB to the heme identifying names. The PDB is {pdb}")
    return heme

def convert_ent_to_pdb(ent_file_path, pdb_file_path):
    parser = PDBParser()
    structure = parser.get_structure('structure', ent_file_path)
    io = Bio.PDB.PDBIO()
    io.set_structure(structure)
    io.save(pdb_file_path)
    os.remove(ent_file_path)
    return pdb_file_path

def HETATM_to_ATM(pdb_id):
    verbose = False
    lines = pdb_tools.files.read_file(pdb_file=f"PDB/{pdb_id}_heme.pdb")
    lines_ = ["CRYST1\n"]
    if verbose:
        print(lines)
    for line in lines:
        line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
        pdb_tools.line_operations.exchange_HETATM_ATOM(line_dict=line_dict)
        line_ = pdb_tools.line_operations.create_line(line_dict=line_dict)
        lines_.append(line_)
    lines_.append("")
    if verbose:
        print(lines_)
    pdb_tools.files.write_file(file=f"PDB/{pdb_id}_heme_ATOM.pdb", lines=lines_)
    try:
        with open(f"PDB/{pdb_id}_heme_ATOM.pdb", 'r') as file:
            content = file.read()
        modified_content = content.replace(' E', 'FE')
        with open(f"PDB/{pdb_id}_heme_ATOM.pdb", 'w') as file:
            file.write(modified_content)
        if verbose:
            print(f"Occurrences of ' E' replaced with 'FE' in 'PDB/{pdb_id}_heme_ATOM.pdb'.")
    except FileNotFoundError:
        print(f"File 'PDB/{pdb_id}_heme_ATOM.pdb' not found.")
    return

def read_pdb(pdb_id, universes, hemes):
    filename = PDB.PDBList().retrieve_pdb_file(pdb_id, pdir='PDB', file_format='pdb')
    filename = convert_ent_to_pdb(ent_file_path=f"PDB/pdb{pdb_id}.ent", pdb_file_path=f"PDB/{pdb_id}.pdb")
    #for offline testing:
    #filename=f"PDB/{self.pdb_id}.pdb"
    universes[pdb_id], hemes[pdb_id] = mda.Universe(filename), isolate_heme(filename)
    HETATM_to_ATM()
    return

def compare_axials(data):
    verbose, cleanup, universes, hemes, residue_names = False, True, {}, {}, []
    for pdb_id in data['# PDB'].iloc[:]:
        read_pdb()
        len_residue_list_before = len(residue_names)
        try:
            get_axial()
            atoms_residues_in_structure = []
            lines = pdb_tools.files.read_file(f"PDB/{pdb_id}_axials.pdb")
            for line in lines:
                line_dict = pdb_tools.line_operations.read_pdb_line(line=line)
                atoms_residues_in_structure.append(line_dict["resname"])
            for atoms_residue in atoms_residues_in_structure:
                if atoms_residue not in residue_names:
                    residue_names.append(atoms_residue)
            len_residue_list_after = len(residue_names)
            if len_residue_list_before < len_residue_list_after:
                with open("axial_ligands.txt", "w") as file:
                    file.write(str(residue_names))
        except IndexError:
            continue
        if cleanup:
            cleaner()
    return

In [25]:
data = pd.read_csv('pyDISH_data.csv')
data = data.drop('Unnamed: 0', axis=1)
result_large = data[["# PDB", "Heme", "ligand"]]
#result_large
result_unique_combinations = result_large["ligand"].unique()
unique_residues_ = [string.split("-") for string in result_unique_combinations]
unique_residues = pd.DataFrame(unique_residues_, columns=["top", "bottom"])
unique_residues_top = pd.DataFrame(unique_residues["top"].unique())
unique_residues_bottom = pd.DataFrame(unique_residues["bottom"].unique())
unique_residues = pd.concat([unique_residues_top, unique_residues_bottom], ignore_index=True)
unique_residues = unique_residues[0].unique()
#unique_residues

In [26]:
unique_combinations_list = ['HIS-NBN', 'HIS-HOH', 'HIS-ENC', 'HIS-MNC', 'HIS-NPN', 'HIS-MET',
       'HIS-HIS', 'HIS', 'TYR-AZI', 'HIS-OXY', 'HIS-CMO', 'HIS-CYN',
       'CYS', 'HIS-AZI', 'HIS-TYR', 'MET-MET', 'HIS-CL', 'HIS-ACT',
       'CYS-HOH', 'HIS-NO', 'CYS-EDO', 'None', 'HIS-HOA', 'HOH-HOH',
       'HOH-IMD', 'TYR', 'CYS-NO', 'HIS-O', 'CYS-BNZ', 'HIS-DOD',
       'HIS-IMD', 'HIS-NH3', 'CYS-1PM', 'TYR-CYN', 'HIS-HGU', 'IMD',
       'PO4', '4MZ-HOH', '1MZ-HOH', 'ASN-HIS', 'CYS-OXY', 'CYS-O', 'NO',
       'TYR-HOH', 'CYS-PIM', 'CYS-TPF', 'CYS-9AP', 'CYS-IMD', 'LYS-HOH',
       'LYS-SO4', 'HIS-NIO', 'HIS-PRO', 'HIS-ISQ', 'CYS-NBN', 'TYR-PEO',
       'HIS-OH', 'TYR-O', 'CYS-HIS', 'HIS-CSS', 'HOH-HSO', 'HIS-HSM',
       'CYS-CMO', 'MET', 'CYS-KTN', 'HIS-F', 'HIS-NBE', 'HIS-1MZ',
       'HIS-H2S', 'CYS-TMH', 'CYS-CYN', 'HOH-OXY', 'TYR-FMT', 'CYS-CAH',
       'HIS-NOE', 'HIS-ACY', 'CYS-NCT', 'CYS-PFZ', 'CYS-PIW', 'CYS-SO4',
       'CYS-MYT', 'HIS-GOL', 'TYR-OXY', 'CYS-GLU', 'LYS-SCN', 'CYS-CPZ',
       'HIS-NH4', 'HIS-ETN', 'TYR-3TR', 'HIS-PPI', 'IMD-IMD', 'HOH-HSM',
       'HIS-SHA', 'HOH', 'HIS-FMT', 'HIS-NZH', 'CYS-COU', 'CYS-TMI',
       'HIS-LYS', 'CYS-1CM', 'CYS-CM6', 'CYS-ACT', 'CYS-FMT', 'HIS-PIM',
       'HIS-224', 'LYS-NO2', 'LYS-HOA', 'HIS-OSM', 'ACY-HOH', 'BME',
       'HIS-SCN', 'CYS-D1G', 'CYS-D2G', 'CYS-D3G', 'CYS-D4G', 'HIS-NO2',
       'HIS-P1R', 'CYS-1MZ', 'CYS-LYS', 'HOH-TPF', 'LYS-MET', 'CYS-DMS',
       'CYS-PEO', 'ASP-MET', 'LYS', 'HIS-PO4', 'CYS-KLN', 'HIS-NSM',
       'CYS-226', 'HIS-PER', 'CYS-228', 'CYS-R86', 'CYS-MSR', 'CYS-333',
       'CYS-391', 'CYS-342', 'GSH', 'HIS-SO4', 'HIS-MSE', 'BCT-IMD',
       'CYS-ECN', 'HIS-TRP', 'HIS-DTR', 'HIS-CTE', 'HIS-DTE', 'HIS-ISZ',
       'CYS-CM9', 'CYS-CII', 'CYS-CMW', 'CYS-II2', 'CYS-II4', 'CYS-X2N',
       'CYS-CL6', 'CYS-PG4', 'CMO-HOH', 'CYS-MZ0', 'AZI-HOH', 'LYS-AZI',
       'HIS-PXO', 'HIS-BXO', 'CYS-VDY', 'CYS-MXD', 'CYS-U51', 'CYS-MLI',
       'CYS-LEH', 'LYS-SO3', 'HIS-AD8', 'CYS-UNX', 'CYS-4PZ', 'CYS-LZ1',
       'CYS-GLN', 'SEC', 'CYS-PB2', 'LYS-PO4', 'CYS-OID', 'CYS-VNI',
       'HIS-PEO', 'HIS-Q80', 'CL', 'HIS-DTV', 'HIS-DTU', 'CYS-JM8',
       'CYS-JM7', 'CYS-JM5', 'CYS-JM4', 'ECL-ECN', 'CYS-POZ', 'HIS-Q86',
       'CYS-OIO', 'CYS-KKK', 'GLU-HIS', 'CYS-FJZ', 'CYS-GJZ', 'CYS-VOR',
       'LYS-CYN', 'TYR-IMD', 'CYS-HC9', 'CYS-RIT', 'HIS-X89', 'HIS-ECN',
       'HIS-KKK', 'CYS-P3Z', 'CYS-PN0', 'CYS-3QO', 'CYS-3QU', 'HIS-PZA',
       'CYS-ECL', 'TYR-NO', 'TYR-NH3', 'HIS-MZZ', 'MET-TYR', 'CYS-AER',
       'MSE-TYR', 'HIS-CSD', 'CYS-TOK', 'CYS-9PL', 'ARG-HIS', 'HIS-3TG',
       'CYS-JKF', 'CYS-D0R', 'CYS-06X', 'HIS-TOX', 'CYS-GOL', 'HIS-AXO',
       'HIS-EPE', 'CYS-UDO', 'CYS-UDD', 'HIS-NCA', 'HIS-SER', 'CYS-18I',
       'CYS-TU1', 'CYS-WVH', 'HIS-NIH', 'CYS-26N', 'CYS-TW5', 'CYS-M87',
       'CYS-ML6', 'CYS-M68', 'CYS-LFD', 'CYS-LFT', 'CYS-LFS', 'CYS-T9H',
       'CYS-L5Z', 'CYS-K9L', 'CYS-EG8', 'CYS-OLW', 'CYS-7F5', 'CYS-6J0',
       'CYS-E2Z', 'CYS-4E8', 'CYS-HLW', 'CYS-9HL', 'CYS-PK9', 'CYS-J9K',
       'CYS-PKT', 'CYS-SRO', 'CYS-LDP', 'CYS-0QA', 'CYS-FVX', 'CYS-0T3',
       'CYS-TQU', 'CYS-VNT', 'CYS-TZM', 'CYS-MQN', 'CYS-VFV', 'CYS-NEE',
       'CYS-SYN', 'HIS-BME', 'HIS-HCS', 'CYS-Z8Z', 'CYS-Z9Z', 'TYR-CL',
       'CYS-1RD', 'CYS-5AW', 'CYS-6AW', 'CYS-7AW', 'CYS-8AW', 'CYS-PGE',
       'HIS-HYP', 'HIS-HZN', 'HIS-PHZ', 'HIS-UNX', 'CYS-2QH', 'CYN-HOH',
       'HIS-PKJ', 'HOH-SMA', 'CYS-KH4', 'TYR-PYG', 'TYR-CAQ', 'HIS-SO2',
       'HIS-SX', 'HIS-SO3', 'CYS-36Y', 'CYS-25S', 'CYS-J5Y', 'CYS-S8F',
       'GLU', 'SER', 'CYS-3SQ', 'GLY-HIS', 'CYS-SI6', 'HIS-UNL', 'ASP',
       'CYS-1YN', 'CYS-2SN', 'CYS-VUR', 'CYS-VT1', 'CYS-1PE', 'CYS-TBX',
       'CYS-TBQ', 'CYS-5L9', 'CYS-5L8', 'CYS-FQC', 'CYS-5LU', '5LW-5LX',
       'HIS-5PJ', 'HIS-5PK', 'HIS-5PF', 'HIS-XNL', 'HIS-MMZ', 'CYS-XEB',
       'CYS-VT2', 'CYS-PLM', 'CYS-12H', 'HIS-MZY', 'HIS-IOD', 'CYS-GLY',
       'HIS-3CJ', 'CYS-GGJ', 'CYS-69M', 'CYS-69W', 'CYS-69S', 'CYS-6D7',
       'CYS-7D6', 'CYS-32M', 'HIS-NVI', 'HIS-6EX', 'HIS-DMS', 'HIS-BZI',
       'CYS-6RJ', 'HIS-IS8', 'HIS-3QM', 'HOH-MHS', 'CYS-C0R', 'CYS-TES',
       'CYS-6XD', 'HIS-0CT', 'CYS-9KE', 'CYS-9KB', 'IMD-MHS', 'HIS-NIZ',
       'CYS-08J', 'HIS-NFK', 'CYS-8J4', 'CYS-8QD', 'CYS-P64', 'CYS-XH8',
       'CYS-9PM', 'CYS-9P1', 'CYS-9P7', 'CYS-P94', 'HIS-BBJ', 'HIS-DAH',
       'CYS-UCZ', 'CYS-9RL', 'CYS-HEZ', 'HIS-PHE', 'HIS-9R9', 'HIS-FER',
       'ALA-HIS', 'CYS-D7Y', 'CYS-D7M', 'CYS-80K', 'CYS-D7J', 'CYS-D81',
       'CYS-DEV', 'CYS-DEJ', 'CYS-DJ1', 'CYS-Y83', 'CYS-V16', 'CYS-3NX',
       'CYS-3NQ', 'CYS-3NR', 'CYS-LFV', 'CYS-2CV', 'HIS-CA1', 'CYS-G0D',
       'CYS-G1J', 'CYS-G0J', 'CYS-G0M', 'CYS-G0V', 'CYS-FZV', 'CYS-G0Y',
       'CYS-G1D', 'HIS-NP0', 'HIS-HQS', 'HIS-HQJ', 'HIS-B5B', 'CYS-VOH',
       'HIS-C82', 'EEE-MHS', 'HIS-EDO', 'CYS-FJQ', 'HIS-ILE', 'CYS-TDZ',
       'CYS-DTT', 'CYS-B9L', 'HIS-DO9', 'HIS-DU6', 'CYS-JD7', 'HIS-LKP',
       'HIS-PCI', 'HIS-OY4', 'CYS-PJM', 'HIS-THR', 'HIS-JTB', 'CYS-M65',
       'CYS-M8N', 'CYS-PQM', 'CYS-PQP', 'CYS-QES', 'CYS-QEP', 'CYS-QEA',
       'CYS-QDV', 'CYS-QDY', 'CYS-QE4', 'CYS-QDJ', 'CYS-QDP', 'CYS-QFD',
       'CYS-QKM', 'CYS-EPE', 'CYS-U7P', 'CYS-UDJ', 'CYS-O4T', 'CYS-O4W',
       'OXY-Q75', 'HIS-RCN', 'HIS-RCQ', 'HIS-RCW', 'CYS-OC9', 'CYS-J6C',
       'CYS-G4O', 'CYS-3CN', 'CYS-HAI', 'CYS-RM1', 'CYS-STQ', 'CYS-QSC',
       'CYS-1Y5', 'CYS-GQR', 'CYS-NEH', 'CYS-NME', 'HIS-HU0', 'HIS-HU3',
       'HIS-HU6', 'HIS-HU9', 'HIS-HUC', 'CYS-98B', 'CYS-HOA', 'HIS-K7M',
       'HIS-PG0', 'CYS-X8P', 'CYS-X5Y', 'CYS-X7P', 'CYS-X7M', 'CYS-X7J',
       'CYS-X71', 'CYS-X6V', 'CYS-X6S', 'CYS-X6J', 'CYS-X7D', 'CYS-X74',
       'CYS-YBS', 'CYS-YCJ', 'CYS-YCG', 'HIS-YRM', 'CYS-04Y', 'CYS-05D',
       'CYS-PGO', 'CYS-MES', 'HIS-MPD', 'CYS-7PW', 'CYS-81H']

In [27]:
unique_axials_list = ['HIS', 'TYR', 'CYS', 'MET', 'None', 'HOH', 'IMD', 'PO4', '4MZ',
       '1MZ', 'ASN', 'NO', 'LYS', 'ACY', 'BME', 'ASP', 'GSH', 'BCT',
       'CMO', 'AZI', 'SEC', 'CL', 'ECL', 'GLU', 'MSE', 'ARG', 'CYN',
       'SER', 'GLY', '5LW', 'ALA', 'EEE', 'OXY', 'NBN', 'ENC', 'MNC',
       'NPN', None, 'ACT', 'EDO', 'HOA', 'O', 'BNZ', 'DOD', 'NH3', '1PM',
       'HGU', 'PIM', 'TPF', '9AP', 'SO4', 'NIO', 'PRO', 'ISQ', 'PEO',
       'OH', 'CSS', 'HSO', 'HSM', 'KTN', 'F', 'NBE', 'H2S', 'TMH', 'FMT',
       'CAH', 'NOE', 'NCT', 'PFZ', 'PIW', 'MYT', 'GOL', 'SCN', 'CPZ',
       'NH4', 'ETN', '3TR', 'PPI', 'SHA', 'NZH', 'COU', 'TMI', '1CM',
       'CM6', '224', 'NO2', 'OSM', 'D1G', 'D2G', 'D3G', 'D4G', 'P1R',
       'DMS', 'KLN', 'NSM', '226', 'PER', '228', 'R86', 'MSR', '333',
       '391', '342', 'ECN', 'TRP', 'DTR', 'CTE', 'DTE', 'ISZ', 'CM9',
       'CII', 'CMW', 'II2', 'II4', 'X2N', 'CL6', 'PG4', 'MZ0', 'PXO',
       'BXO', 'VDY', 'MXD', 'U51', 'MLI', 'LEH', 'SO3', 'AD8', 'UNX',
       '4PZ', 'LZ1', 'GLN', 'PB2', 'OID', 'VNI', 'Q80', 'DTV', 'DTU',
       'JM8', 'JM7', 'JM5', 'JM4', 'POZ', 'Q86', 'OIO', 'KKK', 'FJZ',
       'GJZ', 'VOR', 'HC9', 'RIT', 'X89', 'P3Z', 'PN0', '3QO', '3QU',
       'PZA', 'MZZ', 'AER', 'CSD', 'TOK', '9PL', '3TG', 'JKF', 'D0R',
       '06X', 'TOX', 'AXO', 'EPE', 'UDO', 'UDD', 'NCA', '18I', 'TU1',
       'WVH', 'NIH', '26N', 'TW5', 'M87', 'ML6', 'M68', 'LFD', 'LFT',
       'LFS', 'T9H', 'L5Z', 'K9L', 'EG8', 'OLW', '7F5', '6J0', 'E2Z',
       '4E8', 'HLW', '9HL', 'PK9', 'J9K', 'PKT', 'SRO', 'LDP', '0QA',
       'FVX', '0T3', 'TQU', 'VNT', 'TZM', 'MQN', 'VFV', 'NEE', 'SYN',
       'HCS', 'Z8Z', 'Z9Z', '1RD', '5AW', '6AW', '7AW', '8AW', 'PGE',
       'HYP', 'HZN', 'PHZ', '2QH', 'PKJ', 'SMA', 'KH4', 'PYG', 'CAQ',
       'SO2', 'SX', '36Y', '25S', 'J5Y', 'S8F', '3SQ', 'SI6', 'UNL',
       '1YN', '2SN', 'VUR', 'VT1', '1PE', 'TBX', 'TBQ', '5L9', '5L8',
       'FQC', '5LU', '5LX', '5PJ', '5PK', '5PF', 'XNL', 'MMZ', 'XEB',
       'VT2', 'PLM', '12H', 'MZY', 'IOD', '3CJ', 'GGJ', '69M', '69W',
       '69S', '6D7', '7D6', '32M', 'NVI', '6EX', 'BZI', '6RJ', 'IS8',
       '3QM', 'MHS', 'C0R', 'TES', '6XD', '0CT', '9KE', '9KB', 'NIZ',
       '08J', 'NFK', '8J4', '8QD', 'P64', 'XH8', '9PM', '9P1', '9P7',
       'P94', 'BBJ', 'DAH', 'UCZ', '9RL', 'HEZ', 'PHE', '9R9', 'FER',
       'D7Y', 'D7M', '80K', 'D7J', 'D81', 'DEV', 'DEJ', 'DJ1', 'Y83',
       'V16', '3NX', '3NQ', '3NR', 'LFV', '2CV', 'CA1', 'G0D', 'G1J',
       'G0J', 'G0M', 'G0V', 'FZV', 'G0Y', 'G1D', 'NP0', 'HQS', 'HQJ',
       'B5B', 'VOH', 'C82', 'FJQ', 'ILE', 'TDZ', 'DTT', 'B9L', 'DO9',
       'DU6', 'JD7', 'LKP', 'PCI', 'OY4', 'PJM', 'THR', 'JTB', 'M65',
       'M8N', 'PQM', 'PQP', 'QES', 'QEP', 'QEA', 'QDV', 'QDY', 'QE4',
       'QDJ', 'QDP', 'QFD', 'QKM', 'U7P', 'UDJ', 'O4T', 'O4W', 'Q75',
       'RCN', 'RCQ', 'RCW', 'OC9', 'J6C', 'G4O', '3CN', 'HAI', 'RM1',
       'STQ', 'QSC', '1Y5', 'GQR', 'NEH', 'NME', 'HU0', 'HU3', 'HU6',
       'HU9', 'HUC', '98B', 'K7M', 'PG0', 'X8P', 'X5Y', 'X7P', 'X7M',
       'X7J', 'X71', 'X6V', 'X6S', 'X6J', 'X7D', 'X74', 'YBS', 'YCJ',
       'YCG', 'YRM', '04Y', '05D', 'PGO', 'MES', 'MPD', '7PW', '81H']

In [34]:
duplicated_values = data['# PDB'].duplicated(keep=False)
data = data[~duplicated_values]
result_large = data[["# PDB", "Heme", "ligand"]]
#result_large
result_unique_combinations = result_large["ligand"].unique()
unique_residues_ = [string.split("-") for string in result_unique_combinations]
unique_residues = pd.DataFrame(unique_residues_, columns=["top", "bottom"])
unique_residues_top = pd.DataFrame(unique_residues["top"].unique())
unique_residues_bottom = pd.DataFrame(unique_residues["bottom"].unique())
unique_residues = pd.concat([unique_residues_top, unique_residues_bottom], ignore_index=True)
unique_residues = unique_residues[0].unique()
unique_residues

array(['HIS', 'CYS', 'None', 'HOH', 'IMD', 'PO4', '4MZ', '1MZ', 'NO',
       'TYR', 'ACY', 'BME', 'LYS', 'CMO', 'SEC', 'GLU', 'ARG', 'MET',
       'GLY', '5LW', 'EEE', 'OXY', 'NBN', 'ENC', 'MNC', 'NPN', None,
       'CYN', 'AZI', 'CL', 'HOA', 'BNZ', 'DOD', 'NH3', '1PM', 'HGU',
       'PIM', 'TPF', 'O', '9AP', 'ISQ', 'OH', 'HSM', 'KTN', 'ACT', 'F',
       'NIO', 'NBE', 'H2S', 'FMT', 'CAH', 'NOE', 'NCT', 'PFZ', 'PIW',
       'SO4', 'MYT', 'CPZ', 'NH4', 'ETN', 'PPI', 'SHA', 'NZH', 'TMI',
       '1CM', 'CM6', '224', 'SCN', 'NO2', 'P1R', 'PEO', 'KLN', 'NSM',
       '228', 'R86', 'MSR', '333', '391', '342', 'MSE', 'TRP', 'DTR',
       'CTE', 'DTE', 'ISZ', 'PER', 'CM9', 'CII', 'CMW', 'II4', 'CL6',
       'PG4', 'OSM', 'DTV', 'DTU', 'POZ', 'FJZ', 'X89', 'P3Z', '3QO',
       'PZA', 'ECL', 'MZZ', 'D0R', '06X', 'GOL', 'EPE', 'UDO', 'UDD',
       'EDO', 'WVH', 'LFT', 'LFS', 'PK9', 'J9K', 'PKT', '0QA', 'FVX',
       'TQU', 'TZM', 'MQN', 'NEE', 'HCS', 'Z8Z', 'Z9Z', 'X2N', '1RD',
       '5AW', '6AW',

In [33]:
unique_combinations_single_heme = ['HIS-NBN', 'HIS-HOH', 'HIS-ENC', 'HIS-MNC', 'HIS-NPN', 'HIS-MET',
       'HIS-CMO', 'HIS-OXY', 'HIS', 'CYS', 'HIS-HIS', 'HIS-CYN',
       'HIS-AZI', 'HIS-TYR', 'HIS-CL', 'None', 'HIS-HOA', 'HOH-HOH',
       'HOH-IMD', 'CYS-NO', 'CYS-BNZ', 'CYS-HOH', 'HIS-DOD', 'HIS-IMD',
       'HIS-NH3', 'CYS-1PM', 'HIS-HGU', 'HIS-NO', 'IMD', 'PO4', '4MZ-HOH',
       '1MZ-HOH', 'NO', 'TYR-HOH', 'CYS-PIM', 'CYS-TPF', 'HIS-O',
       'CYS-9AP', 'HIS-ISQ', 'CYS-NBN', 'HIS-OH', 'TYR-O', 'TYR',
       'HIS-HSM', 'CYS-CMO', 'CYS-KTN', 'HIS-ACT', 'HIS-F', 'HIS-NIO',
       'HIS-NBE', 'HIS-1MZ', 'HIS-H2S', 'HOH-OXY', 'TYR-FMT', 'CYS-IMD',
       'CYS-CAH', 'HIS-NOE', 'CYS-NCT', 'CYS-PFZ', 'CYS-PIW', 'CYS-SO4',
       'CYS-MYT', 'CYS-CPZ', 'HIS-NH4', 'HIS-ETN', 'HIS-PPI', 'HIS-SHA',
       'HIS-FMT', 'HIS-NZH', 'CYS-OXY', 'CYS-TMI', 'HIS-LYS', 'CYS-1CM',
       'CYS-CM6', 'CYS-CYN', 'CYS-ACT', 'CYS-FMT', 'HIS-224', 'ACY-HOH',
       'BME', 'HIS-SCN', 'HIS-NO2', 'HIS-P1R', 'CYS-1MZ', 'LYS-MET',
       'CYS-PEO', 'HIS-PO4', 'CYS-KLN', 'HIS-NSM', 'CYS-228', 'CYS-R86',
       'CYS-MSR', 'CYS-333', 'CYS-391', 'CYS-342', 'HIS-MSE', 'HIS-TRP',
       'HIS-DTR', 'HIS-CTE', 'HIS-DTE', 'HIS-ISZ', 'HIS-PER', 'CYS-CM9',
       'CYS-CII', 'CYS-CMW', 'CYS-II4', 'CYS-CL6', 'CYS-PG4', 'CMO-HOH',
       'HIS-OSM', 'CYS-HIS', 'SEC', 'HIS-PEO', 'HIS-DTV', 'HIS-DTU',
       'CYS-POZ', 'GLU-HIS', 'CYS-FJZ', 'HIS-X89', 'CYS-P3Z', 'CYS-3QO',
       'HIS-PZA', 'CYS-ECL', 'HIS-MZZ', 'ARG-HIS', 'CYS-D0R', 'CYS-06X',
       'CYS-GOL', 'HIS-EPE', 'CYS-UDO', 'CYS-UDD', 'MET-MET', 'CYS-EDO',
       'CYS-WVH', 'CYS-LFT', 'CYS-LFS', 'CYS-PK9', 'CYS-J9K', 'CYS-PKT',
       'CYS-0QA', 'CYS-FVX', 'CYS-TQU', 'CYS-TZM', 'CYS-MQN', 'CYS-NEE',
       'MET-TYR', 'HIS-BME', 'HIS-HCS', 'CYS-Z8Z', 'CYS-Z9Z', 'CYS-X2N',
       'CYS-1RD', 'CYS-5AW', 'CYS-6AW', 'CYS-8AW', 'CYS-PGE', 'CYS-2QH',
       'CYS-KH4', 'CYS-36Y', 'CYS-25S', 'CYS-J5Y', 'GLY-HIS', 'CYS-1YN',
       'CYS-VOR', 'CYS-VT1', 'CYS-1PE', 'CYS-TBX', 'CYS-TBQ', 'CYS-5L9',
       'CYS-5L8', 'CYS-5LU', '5LW-5LX', 'CYS-VT2', 'HIS-MZY', 'HIS-IOD',
       'CYS-GGJ', 'CYS-69M', 'CYS-69W', 'CYS-69S', 'CYS-32M', 'CYS-6RJ',
       'HIS-3QM', 'CYS-KKK', 'CYS-9KE', 'CYS-9KB', 'HOH-MHS', 'IMD-MHS',
       'CYS-DMS', 'CYS-RIT', 'CYS-UCZ', 'CYS-9RL', 'CYS-HEZ', 'HIS-FER',
       'CYS-D7Y', 'CYS-D7M', 'CYS-80K', 'CYS-D7J', 'CYS-D81', 'CYS-DEV',
       'CYS-DEJ', 'CYS-DJ1', 'CYS-Y83', 'CYS-V16', 'HIS-CA1', 'CYS-G0D',
       'CYS-G1J', 'CYS-G0J', 'CYS-G0M', 'CYS-G0V', 'CYS-FZV', 'CYS-G0Y',
       'CYS-G1D', 'HIS-NP0', 'CYS-SYN', 'CYS-VOH', 'EEE-MHS', 'HIS-EDO',
       'CYS-M65', 'CYS-M8N', 'CYS-PQM', 'CYS-PQP', 'CYS-QES', 'CYS-QEP',
       'CYS-QEA', 'CYS-QDV', 'CYS-QE4', 'CYS-QDJ', 'CYS-QFD', 'CYS-EPE',
       'HOH', 'OXY-Q75', 'CYS-J6C', 'GLU', 'CYS-X8P', 'CYS-X7P',
       'CYS-X7M', 'CYS-X71', 'CYS-X6J', 'CYS-YBS', 'CYS-YCJ', 'CYS-YCG',
       'CYS-04Y', 'CYS-05D', 'CYS-PGO', 'CYS-MES', 'CYS-81H']
len(unique_combinations_single_heme)

246

In [35]:
unique_residues_single_heme = ['HIS', 'CYS', 'None', 'HOH', 'IMD', 'PO4', '4MZ', '1MZ', 'NO',
       'TYR', 'ACY', 'BME', 'LYS', 'CMO', 'SEC', 'GLU', 'ARG', 'MET',
       'GLY', '5LW', 'EEE', 'OXY', 'NBN', 'ENC', 'MNC', 'NPN', None,
       'CYN', 'AZI', 'CL', 'HOA', 'BNZ', 'DOD', 'NH3', '1PM', 'HGU',
       'PIM', 'TPF', 'O', '9AP', 'ISQ', 'OH', 'HSM', 'KTN', 'ACT', 'F',
       'NIO', 'NBE', 'H2S', 'FMT', 'CAH', 'NOE', 'NCT', 'PFZ', 'PIW',
       'SO4', 'MYT', 'CPZ', 'NH4', 'ETN', 'PPI', 'SHA', 'NZH', 'TMI',
       '1CM', 'CM6', '224', 'SCN', 'NO2', 'P1R', 'PEO', 'KLN', 'NSM',
       '228', 'R86', 'MSR', '333', '391', '342', 'MSE', 'TRP', 'DTR',
       'CTE', 'DTE', 'ISZ', 'PER', 'CM9', 'CII', 'CMW', 'II4', 'CL6',
       'PG4', 'OSM', 'DTV', 'DTU', 'POZ', 'FJZ', 'X89', 'P3Z', '3QO',
       'PZA', 'ECL', 'MZZ', 'D0R', '06X', 'GOL', 'EPE', 'UDO', 'UDD',
       'EDO', 'WVH', 'LFT', 'LFS', 'PK9', 'J9K', 'PKT', '0QA', 'FVX',
       'TQU', 'TZM', 'MQN', 'NEE', 'HCS', 'Z8Z', 'Z9Z', 'X2N', '1RD',
       '5AW', '6AW', '8AW', 'PGE', '2QH', 'KH4', '36Y', '25S', 'J5Y',
       '1YN', 'VOR', 'VT1', '1PE', 'TBX', 'TBQ', '5L9', '5L8', '5LU',
       '5LX', 'VT2', 'MZY', 'IOD', 'GGJ', '69M', '69W', '69S', '32M',
       '6RJ', '3QM', 'KKK', '9KE', '9KB', 'MHS', 'DMS', 'RIT', 'UCZ',
       '9RL', 'HEZ', 'FER', 'D7Y', 'D7M', '80K', 'D7J', 'D81', 'DEV',
       'DEJ', 'DJ1', 'Y83', 'V16', 'CA1', 'G0D', 'G1J', 'G0J', 'G0M',
       'G0V', 'FZV', 'G0Y', 'G1D', 'NP0', 'SYN', 'VOH', 'M65', 'M8N',
       'PQM', 'PQP', 'QES', 'QEP', 'QEA', 'QDV', 'QE4', 'QDJ', 'QFD',
       'Q75', 'J6C', 'X8P', 'X7P', 'X7M', 'X71', 'X6J', 'YBS', 'YCJ',
       'YCG', '04Y', '05D', 'PGO', 'MES', '81H']
len(unique_residues_single_heme)

214

In [21]:
import ast
with open("axial_ligands.txt", "r") as file:
    line = file.readline()
lines = ast.literal_eval(line)
lines = [k.strip() for k in lines]
lines_set = set(lines)
lines_list = list(lines_set)
print("Set length")
print(len(lines_list))
print("original length")
print(len(lines))

Set length
815
original length
1530
